# Agoda Review Data Preparation

Two-step pipeline:

1. **`agoda-review-prepare.csv`** — full 197k reviews enriched with parsed stay detail (`stay_nights`, `stay_month`, `stay_year`, `stay_period`) and distance-to-coast from `hotel_with_distance.csv`
2. **`agoda-review-en-vi.csv`** — filtered to English + Vietnamese only, columns trimmed for topic modeling

**Prerequisites:** run `pre-scraping/data-prepare.ipynb` first to generate `data/hotel_with_distance.csv`

In [1]:
import re
import pandas as pd
from pathlib import Path

DATA_DIR = Path("../../data")
OUT_DIR  = Path(".")

REVIEWS_PATH  = DATA_DIR / "agoda-reviews.csv"
DISTANCE_PATH = DATA_DIR / "hotel_with_distance.csv"
PREPARE_OUT   = OUT_DIR  / "agoda-review-prepare.csv"
EN_VI_OUT     = OUT_DIR  / "agoda-review-en-vi.csv"

## 1. Load Agoda Reviews

In [2]:
df = pd.read_csv(REVIEWS_PATH, encoding="utf-8-sig", low_memory=False)
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
df.head(2)

Shape: (197508, 25)
Columns: ['hotel_id', 'hotel_name', 'addressline1', 'city', 'state', 'numberrooms', 'yearopened', 'yearrenovated', 'number_of_reviews', 'rating_average', 'star_rating', 'national', 'groupname', 'staydetail', 'reviewtitle', 'comment', 'positive', 'negative', 'score', 'accommodationtype1', 'accommodationtype2', 'accommodationtype3', 'accommodationtype4', 'language', 'reviewer_continent']


,hotel_id,hotel_name,addressline1,city,state,numberrooms,yearopened,yearrenovated,number_of_reviews,rating_average,...,comment,positive,negative,score,accommodationtype1,accommodationtype2,accommodationtype3,accommodationtype4,language,reviewer_continent
0,163,Ramana Saigon Hotel,"323 Le Van Sy Street, District 3",Ho Chi Minh City,Ho Chi Minh,296.0,1996.0,2015.0,1713,8.1,...,A tourist class hotel with very basic amenitie...,NaN,NaN,6.4,Hotel,NaN,NaN,NaN,en,Asia
1,163,Ramana Saigon Hotel,"323 Le Van Sy Street, District 3",Ho Chi Minh City,Ho Chi Minh,296.0,1996.0,2015.0,1713,8.1,...,The hotel rooms etc are good but very very sti...,NaN,NaN,7.6,Hotel,NaN,NaN,NaN,en,Asia


In [3]:
print("Language distribution:")
print(df["language"].value_counts())

Language distribution:
language
en         86322
ja         40634
vi         39025
zh          8975
unknown     5507
fr          3606
de          2837
th          2249
hi          1843
sw          1605
ru          1568
ur           813
es           778
nl           648
it           478
pt           161
tr           152
ar           119
pl           115
bg            61
el            12
Name: count, dtype: int64


## 2. Parse Stay Detail

Column format: `"Đã ở X đêm vào Tháng M năm YYYY"`  
Extracts: `stay_nights`, `stay_month` (1–12), `stay_year`, `stay_period` ("YYYY-MM")

In [4]:
_STAY_PATTERN = re.compile(
    r'Đã ở (\d+) đêm vào Tháng (\d+) năm (\d{4})',
    flags=re.IGNORECASE
)

def parse_stay_detail(text):
    """Returns (nights, month, year) or (None, None, None) if no match."""
    if pd.isna(text):
        return None, None, None
    m = _STAY_PATTERN.search(str(text))
    if not m:
        return None, None, None
    return int(m.group(1)), int(m.group(2)), int(m.group(3))

parsed = df["staydetail"].apply(parse_stay_detail)
df["stay_nights"] = parsed.apply(lambda x: x[0])
df["stay_month"]  = parsed.apply(lambda x: x[1])
df["stay_year"]   = parsed.apply(lambda x: x[2])
df["stay_period"] = df.apply(
    lambda r: f"{int(r.stay_year):04d}-{int(r.stay_month):02d}"
              if pd.notna(r.stay_month) else None,
    axis=1
)

failed = df["stay_month"].isna().sum()
print(f"Parsed: {df['stay_month'].notna().sum():,}  |  Failed: {failed:,}")
df[["staydetail", "stay_nights", "stay_month", "stay_year", "stay_period"]].head(5)

Parsed: 196,531  |  Failed: 977


,staydetail,stay_nights,stay_month,stay_year,stay_period
0,Đã ở 4 đêm vào Tháng 1 năm 2024,4.0,1.0,2024.0,2024-01
1,Đã ở 2 đêm vào Tháng 1 năm 2024,2.0,1.0,2024.0,2024-01
2,Đã ở 4 đêm vào Tháng 4 năm 2015,4.0,4.0,2015.0,2015-04
3,Đã ở 2 đêm vào Tháng 2 năm 2024,2.0,2.0,2024.0,2024-02
4,Đã ở 1 đêm vào Tháng 12 năm 2014,1.0,12.0,2014.0,2014-12


## 3. Enrich with Distance to Coast

Left-join `hotel_with_distance.csv` on `hotel_id` to add `distance2coastline`, `hotel_coordinate`, `nearest_coordinate`.  
Only pull columns not already present in `agoda-reviews.csv`.

In [5]:
dist = pd.read_csv(
    DISTANCE_PATH,
    encoding="utf-8-sig",
    usecols=["hotel_id", "distance2coastline", "hotel_coordinate", "nearest_coordinate"],
)
print("hotel_with_distance loaded:", dist.shape)
dist.head(2)

hotel_with_distance loaded: (8574, 4)


,hotel_id,hotel_coordinate,distance2coastline,nearest_coordinate
0,163,POINT (106.678101 10.787597),47.465,POINT (11907922.454514029 1165509.8719933222)
1,902,POINT (106.706797 10.778689),44.573,POINT (11907922.454514029 1165509.8719933222)


In [6]:
df = df.merge(dist, on="hotel_id", how="left")

missing_dist = df["distance2coastline"].isna().sum()
print(f"Shape after merge  : {df.shape}")
print(f"Hotels with distance data : {len(df) - missing_dist:,} / {len(df):,}")
print(f"Missing distance          : {missing_dist:,}")

df[["hotel_id", "hotel_name", "distance2coastline", "hotel_coordinate"]].head(3)

Shape after merge  : (197508, 32)
Hotels with distance data : 197,508 / 197,508
Missing distance          : 0


,hotel_id,hotel_name,distance2coastline,hotel_coordinate
0,163,Ramana Saigon Hotel,47.465,POINT (106.678101 10.787597)
1,163,Ramana Saigon Hotel,47.465,POINT (106.678101 10.787597)
2,163,Ramana Saigon Hotel,47.465,POINT (106.678101 10.787597)


## 4. Save `agoda-review-prepare.csv`

Full enriched dataset — all languages, all columns.

In [7]:
df.to_csv(PREPARE_OUT, index=False, encoding="utf-8-sig")
print(f"Saved → {PREPARE_OUT}")
print(f"Shape : {df.shape}")
print(f"Columns ({len(df.columns)}): {df.columns.tolist()}")

Saved → agoda-review-prepare.csv
Shape : (197508, 32)
Columns (32): ['hotel_id', 'hotel_name', 'addressline1', 'city', 'state', 'numberrooms', 'yearopened', 'yearrenovated', 'number_of_reviews', 'rating_average', 'star_rating', 'national', 'groupname', 'staydetail', 'reviewtitle', 'comment', 'positive', 'negative', 'score', 'accommodationtype1', 'accommodationtype2', 'accommodationtype3', 'accommodationtype4', 'language', 'reviewer_continent', 'stay_nights', 'stay_month', 'stay_year', 'stay_period', 'hotel_coordinate', 'distance2coastline', 'nearest_coordinate']


## 5. Filter to English + Vietnamese

Keep `language in ['en', 'vi']` and select columns relevant to topic modeling.  
Drop near-empty columns (`positive`, `negative` — 90%+ null) and photo/URL fields.

In [8]:
en_vi = df[df["language"].isin(["en", "vi"])].copy()

print(f"All reviews      : {len(df):,}")
print(f"en + vi reviews  : {len(en_vi):,}")
print()
print("Language breakdown:")
print(en_vi["language"].value_counts())

All reviews      : 197,508
en + vi reviews  : 125,347

Language breakdown:
language
en    86322
vi    39025
Name: count, dtype: int64


In [9]:
# Columns to keep for topic modeling
TOPIC_COLS = [
    # Review text
    "comment",
    "reviewtitle",
    # Review metadata
    "language",
    "score",
    # Stay timing
    "stay_year",
    "stay_month",
    "stay_period",
    "stay_nights",
    # Reviewer context
    "national",
    "reviewer_continent",
    "groupname",
    # Hotel identity
    "hotel_id",
    "hotel_name",
    "city",
    "state",
    # Hotel attributes
    "star_rating",
    "accommodationtype1",
    "numberrooms",
    "yearopened",
    "yearrenovated",
    # Coastal proximity
    "distance2coastline",
    "hotel_coordinate",
]

# Only keep columns that actually exist
topic_cols_available = [c for c in TOPIC_COLS if c in en_vi.columns]
missing_cols = [c for c in TOPIC_COLS if c not in en_vi.columns]
if missing_cols:
    print("Warning — columns not found (skipped):", missing_cols)

en_vi = en_vi[topic_cols_available].reset_index(drop=True)
print(f"\nFinal shape: {en_vi.shape}")
en_vi.head(3)


Final shape: (125347, 22)


,comment,reviewtitle,language,score,stay_year,stay_month,stay_period,stay_nights,national,reviewer_continent,...,hotel_name,city,state,star_rating,accommodationtype1,numberrooms,yearopened,yearrenovated,distance2coastline,hotel_coordinate
0,A tourist class hotel with very basic amenitie...,HCM Short Trip”,en,6.4,2024.0,1.0,2024-01,4.0,Singapore,Asia,...,Ramana Saigon Hotel,Ho Chi Minh City,Ho Chi Minh,4.0,Hotel,296.0,1996.0,2015.0,47.465,POINT (106.678101 10.787597)
1,The hotel rooms etc are good but very very sti...,Hospitality low ”,en,7.6,2024.0,1.0,2024-01,2.0,Ấn Độ,Asia,...,Ramana Saigon Hotel,Ho Chi Minh City,Ho Chi Minh,4.0,Hotel,296.0,1996.0,2015.0,47.465,POINT (106.678101 10.787597)
2,Really enjoy my stay here. Will stay again,Great safe hotel in good location. ”,en,8.7,2015.0,4.0,2015-04,4.0,Canada,North America,...,Ramana Saigon Hotel,Ho Chi Minh City,Ho Chi Minh,4.0,Hotel,296.0,1996.0,2015.0,47.465,POINT (106.678101 10.787597)


In [10]:
# Null audit on key columns
print("Null counts for key columns:")
print(en_vi[["comment", "reviewtitle", "score", "stay_period", "distance2coastline"]].isna().sum())

Null counts for key columns:
comment               0
reviewtitle           0
score                 0
stay_period           0
distance2coastline    0
dtype: int64


In [11]:
# Distribution checks
print("Stay period distribution (last 12):")
print(en_vi["stay_period"].value_counts().sort_index().tail(12))
print()
print("Score distribution:")
print(en_vi["score"].value_counts().sort_index())

Stay period distribution (last 12):
stay_period
2023-05    1918
2023-06    1948
2023-07    2182
2023-08    2277
2023-09    2300
2023-10    1985
2023-11    2123
2023-12    3073
2024-01    2471
2024-02    2999
2024-03    2783
2024-04     575
Name: count, dtype: int64

Score distribution:
score
0.0         3
2.0      1749
2.3        34
2.4       469
2.7        61
2.8       778
3.0        83
3.2       783
3.3       117
3.6      1061
3.7       132
4.0      1344
4.3       214
4.4      1241
4.7       245
4.8      1362
5.0       344
5.2      1613
5.3       411
5.6      1699
5.7       530
6.0      3702
6.3       809
6.4      2508
6.5         2
6.7       979
6.8      3014
7.0      1135
7.2      3540
7.3      1443
7.5         2
7.6      4351
7.7      1762
8.0      9578
8.3      1886
8.4      6385
8.5         3
8.7      1816
8.8      6853
9.0      1757
9.2      8875
9.3      1922
9.5         1
9.6     10856
9.7      1662
10.0    36233
Name: count, dtype: int64


## 6. Save `agoda-review-en-vi.csv`

Topic modeling ready — English + Vietnamese only, trimmed columns.

In [12]:
en_vi.to_csv(EN_VI_OUT, index=False, encoding="utf-8-sig")
print(f"Saved → {EN_VI_OUT}")
print(f"Shape : {en_vi.shape}")
print(f"Columns: {en_vi.columns.tolist()}")

Saved → agoda-review-en-vi.csv
Shape : (125347, 22)
Columns: ['comment', 'reviewtitle', 'language', 'score', 'stay_year', 'stay_month', 'stay_period', 'stay_nights', 'national', 'reviewer_continent', 'groupname', 'hotel_id', 'hotel_name', 'city', 'state', 'star_rating', 'accommodationtype1', 'numberrooms', 'yearopened', 'yearrenovated', 'distance2coastline', 'hotel_coordinate']
